# StreamLoop — Ajuste del modelo de cancelación

Flujo: **split → línea base (test) → Randomized (CV en train) → Grid (CV en train) → `cv_results_` → modelo final (test)**.

⚠️ El test set se toca **exactamente dos veces**: línea base (sección 4) y modelo final (sección 10). Todo lo demás usa solo `X_train`.

## 0. Imports y configuración

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import randint

from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, fbeta_score, recall_score, precision_score, f1_score, accuracy_score

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

RANDOM_STATE = 42
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

## 1. Carga y limpieza mínima

- `TotalCharges` llega como texto con espacios vacíos → se convierte a número (los vacíos pasan a `NaN`).
- `customerID` es un identificador sin valor predictivo → se elimina.
- `Churn` (Yes/No) → 1/0.

No se imputa aquí: la imputación vive dentro del `Pipeline` para que se ajuste solo con datos de entrenamiento.

In [2]:
df = pd.read_csv(URL)
print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("TotalCharges faltantes tras la conversión:", df["TotalCharges"].isna().sum())

df = df.drop(columns="customerID")
df["Churn"] = (df["Churn"] == "Yes").astype(int)

print(f"Tasa de churn: {df['Churn'].mean():.1%}")
df.dtypes.value_counts()

TotalCharges faltantes tras la conversión: 11
Tasa de churn: 26.5%


object     15
int64       3
float64     2
Name: count, dtype: int64

## 2. Split train/test (antes de todo lo demás)

Estratificado por `Churn` (la clase positiva es ~27%) para que train y test mantengan la misma proporción.

In [4]:
X = df.drop(columns="Churn")
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)
print(f"Churn train: {y_train.mean():.1%} | test: {y_test.mean():.1%}")

(5634, 19) (1409, 19)
Churn train: 26.5% | test: 26.5%


## 3. Pipeline con preprocesamiento

- Numéricas: imputación por mediana + `StandardScaler`.
- Categóricas: imputación por moda + `OneHotEncoder(handle_unknown='ignore')`.
- Clasificador: `RandomForestClassifier`.

In [5]:
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()
print("Numéricas:", num_cols)
print("Categóricas:", len(cat_cols))

preprocess = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

pipe = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE)),
])
pipe

Numéricas: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categóricas: 15


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['SeniorCitizen', 'tenure',
                                                   'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['gender', 'Partner',
                                                   'Dependents', 'PhoneService',
                                                   'MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                                                   'TechSupport', 'StreamingTV',
                                                   'StreamingMovies',
                                                   'Contract',
                                                   'PaperlessBilling',
                                                   'PaymentMethod'])])),
                ('clf', RandomForestClassifier(random_state=42))])

## 4. Línea base (hiperparámetros por defecto) — 🔒 1er uso del test set

Se reportan las mismas métricas que se usarán al final: recall, precision, F1, **F2** y accuracy (todas sobre la clase churn).

In [6]:
def evaluate(model, X_eval, y_eval):
    pred = model.predict(X_eval)
    return {
        "recall": recall_score(y_eval, pred),
        "precision": precision_score(y_eval, pred),
        "f1": f1_score(y_eval, pred),
        "f2": fbeta_score(y_eval, pred, beta=2),
        "accuracy": accuracy_score(y_eval, pred),
    }

pipe.fit(X_train, y_train)
baseline_metrics = evaluate(pipe, X_test, y_test)   # <- 1er y único uso del test para la línea base
pd.Series(baseline_metrics, name="baseline").round(4)

recall       0.4759
precision    0.6117
f1           0.5353
f2           0.4980
accuracy     0.7807
Name: baseline, dtype: float64

## 5. Métrica de negocio (`scoring`): F2

Para StreamLoop, **un cliente que cancela sin ser detectado cuesta mucho más que una retención ofrecida sin necesidad**. Es decir, un falso negativo es más caro que un falso positivo.

- El **accuracy** (por defecto) es engañoso: con ~73% de clientes que no cancelan, predecir siempre "no churn" ya da ~73%.
- El **recall** puro se puede inflar prediciendo "churn" para todos.
- **F2** combina precision y recall pero pesa el recall el doble (β=2), que es justo la prioridad del negocio sin permitir ese truco.

Se usa el mismo `StratifiedKFold` en las dos búsquedas para que sean comparables.

In [7]:
SCORING = make_scorer(fbeta_score, beta=2)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 6. Búsqueda amplia — `RandomizedSearchCV` (solo `X_train`)

In [8]:
param_dist = {
    "clf__n_estimators": randint(100, 600),
    "clf__max_depth": [None, 5, 8, 12, 16, 24],
    "clf__min_samples_split": randint(2, 20),
    "clf__min_samples_leaf": randint(1, 10),
    "clf__max_features": ["sqrt", "log2", 0.3, 0.5],
    "clf__class_weight": [None, "balanced", "balanced_subsample"],
}

random_search = RandomizedSearchCV(
    pipe, param_dist, n_iter=40, scoring=SCORING, cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE,
)
random_search.fit(X_train, y_train)

print("Mejor F2 (CV):", round(random_search.best_score_, 4))
random_search.best_params_

Mejor F2 (CV): 0.7221


{'clf__class_weight': 'balanced',
 'clf__max_depth': 5,
 'clf__max_features': 0.3,
 'clf__min_samples_leaf': 1,
 'clf__min_samples_split': 6,
 'clf__n_estimators': 212}

In [9]:
def top_results(search, n=10):
    res = pd.DataFrame(search.cv_results_)
    cols = [c for c in res.columns if c.startswith("param_")] + ["mean_test_score", "std_test_score", "rank_test_score"]
    out = res.sort_values("rank_test_score")[cols].head(n)
    out.columns = [c.replace("param_clf__", "") for c in out.columns]
    return out

top_results(random_search)

,class_weight,max_depth,max_features,min_samples_leaf,min_samples_split,n_estimators,mean_test_score,std_test_score,rank_test_score
34,balanced,5,0.3,1,6,212,0.722126,0.017489,1
3,balanced,5,0.5,5,2,559,0.720304,0.017034,2
32,balanced_subsample,5,0.5,8,16,414,0.719507,0.015267,3
24,balanced,5,0.5,3,8,340,0.719236,0.016449,4
5,balanced_subsample,5,0.5,3,13,154,0.718482,0.016186,5
19,balanced_subsample,5,sqrt,7,10,427,0.715555,0.022443,6
37,balanced,5,sqrt,6,5,538,0.714618,0.020741,7
26,balanced_subsample,8,sqrt,4,14,259,0.711368,0.021288,8
38,balanced_subsample,8,sqrt,6,6,471,0.709456,0.018117,9
1,balanced_subsample,8,0.3,8,5,459,0.704768,0.023002,10


**Lectura del Randomized:** todos los mejores candidatos usan `class_weight` balanceado y `max_depth=5`, que es el **valor más bajo** del rango explorado → conviene refinar hacia profundidades menores. `max_features` 0.3–0.5 domina sobre `sqrt`/`log2`, y `min_samples_split`/`n_estimators` casi no discriminan.

## 7. Búsqueda focalizada — `GridSearchCV` (solo `X_train`)

Se acota el espacio alrededor de lo que funcionó. `min_samples_split` se fija en 6 (el mejor del Randomized, poco influyente).

**Selección con `refit` callable:** en lugar de quedarnos siempre con el mejor promedio, `refit` recibe `cv_results_` y devuelve el índice del candidato elegido; `GridSearchCV` reentrena ese estimador por nosotros (seguimos sin reentrenar `best_estimator_` a mano). Regla: entre los candidatos a ≤ 0.005 del mejor promedio, elegir el de **menor desviación entre folds**.

In [10]:
param_grid = {
    "clf__class_weight": ["balanced", "balanced_subsample"],
    "clf__max_depth": [3, 4, 5, 6, 8],
    "clf__max_features": [0.3, 0.5, 0.7],
    "clf__min_samples_leaf": [1, 3, 5],
    "clf__min_samples_split": [6],
    "clf__n_estimators": [200, 400],
}

TOL = 0.005

def refit_stable(cv_results, tol=TOL):
    mean = np.array(cv_results["mean_test_score"])
    std = np.array(cv_results["std_test_score"])
    eligible = np.where(mean >= mean.max() - tol)[0]
    return int(eligible[np.argmin(std[eligible])])

grid_search = GridSearchCV(
    pipe, param_grid, scoring=SCORING, cv=cv,
    n_jobs=-1, refit=refit_stable,
)
grid_search.fit(X_train, y_train)

print("Candidatos evaluados:", len(grid_search.cv_results_["params"]))
grid_search.best_params_

Candidatos evaluados: 180


{'clf__class_weight': 'balanced',
 'clf__max_depth': 3,
 'clf__max_features': 0.7,
 'clf__min_samples_leaf': 1,
 'clf__min_samples_split': 6,
 'clf__n_estimators': 400}

## 8. Análisis de `cv_results_`: promedio vs. estabilidad

No basta el mejor promedio: se miran la desviación entre folds y los scores de cada fold de los mejores candidatos.

In [11]:
res = pd.DataFrame(grid_search.cv_results_)
split_cols = [f"split{i}_test_score" for i in range(cv.get_n_splits())]
view_cols = ["mean_test_score", "std_test_score"] + split_cols + ["rank_test_score"]

best_mean_idx = res["mean_test_score"].idxmax()
chosen_idx = grid_search.best_index_

top = res.sort_values("mean_test_score", ascending=False).head(10).copy()
top["params"] = top["params"].apply(lambda p: {k.replace("clf__", ""): v for k, v in p.items() if k != "clf__min_samples_split"})
top["elegido"] = np.where(top.index == chosen_idx, "<-- elegido", "")
top[["params"] + view_cols + ["elegido"]].round(4).to_string(index=False)

"                                                                                                                 params  mean_test_score  std_test_score  split0_test_score  split1_test_score  split2_test_score  split3_test_score  split4_test_score  rank_test_score     elegido\n{'class_weight': 'balanced_subsample', 'max_depth': 3, 'max_features': 0.7, 'min_samples_leaf': 1, 'n_estimators': 200}           0.7252          0.0159             0.7181             0.7092             0.7346             0.7517             0.7121                1            \n{'class_weight': 'balanced_subsample', 'max_depth': 3, 'max_features': 0.7, 'min_samples_leaf': 3, 'n_estimators': 200}           0.7252          0.0159             0.7181             0.7092             0.7346             0.7517             0.7121                1            \n{'class_weight': 'balanced_subsample', 'max_depth': 3, 'max_features': 0.7, 'min_samples_leaf': 5, 'n_estimators': 200}           0.7252          0.0159             

In [12]:
# Candidato de mayor promedio vs. candidato elegido por estabilidad
comparison = res.loc[[best_mean_idx, chosen_idx], ["params"] + view_cols].copy()
comparison.index = ["mayor promedio", "elegido (estable)"] if best_mean_idx != chosen_idx else ["mayor promedio = elegido"]
print("Rango de F2 entre folds (max - min):")
for name, i in [("mayor promedio", best_mean_idx), ("elegido", chosen_idx)]:
    s = res.loc[i, split_cols].astype(float)
    print(f"  {name:15s} mean={res.loc[i,'mean_test_score']:.4f} std={res.loc[i,'std_test_score']:.4f} rango={s.max()-s.min():.4f}")
comparison.drop(columns="params").round(4)

Rango de F2 entre folds (max - min):
  mayor promedio  mean=0.7252 std=0.0159 rango=0.0425
  elegido         mean=0.7245 std=0.0141 rango=0.0360


,mean_test_score,std_test_score,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,rank_test_score
mayor promedio,0.7252,0.0159,0.7181,0.7092,0.7346,0.7517,0.7121,1
elegido (estable),0.7245,0.0141,0.7176,0.7118,0.7338,0.7477,0.7117,7


## 9. Selección del modelo final

El modelo final es `grid_search.best_estimator_`, ya reentrenado por `GridSearchCV` (`refit`) sobre todo `X_train`. La justificación (mayor promedio vs. más estable) está en `tuning_report.md`.

In [13]:
final_model = grid_search.best_estimator_
print({k.replace("clf__", ""): v for k, v in grid_search.best_params_.items()})

{'class_weight': 'balanced', 'max_depth': 3, 'max_features': 0.7, 'min_samples_leaf': 1, 'min_samples_split': 6, 'n_estimators': 400}


## 10. Evaluación final — 🔒 2º y último uso del test set

In [14]:
tuned_metrics = evaluate(final_model, X_test, y_test)   # <- 2º y último uso del test

## 11. Comparación línea base vs. modelo ajustado

In [15]:
comp = pd.DataFrame([baseline_metrics, tuned_metrics], index=["baseline", "ajustado"])
comp.loc["mejora"] = comp.loc["ajustado"] - comp.loc["baseline"]
comp.round(4)

,recall,precision,f1,f2,accuracy
baseline,0.4759,0.6117,0.5353,0.4980,0.7807
ajustado,0.8128,0.4795,0.6032,0.7136,0.7161
mejora,0.3369,-0.1322,0.0678,0.2156,-0.0646


Informe completo en [`tuning_report.md`](tuning_report.md).